# Defense vs. position: does it predict beyond the projection?

Issue #132, under the "what is actually predictive" epic (#114). `defense_vs_position` (#120)
carries three recency windows of "points this defense has allowed to this position" — season to
date, last 3 games, last 5 games. Nothing has measured whether any of them tell a manager anything
a projection doesn't already know. This notebook runs them through `weekly_backtest.score_signal`
(#131), the harness built for exactly this question, and answers the five questions the ticket asks
in order.

In [1]:
# Find the repo root from wherever the kernel started, so `src` imports work.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.query import q
from src.gold.weekly_backtest import score_signal
from src.gold.league_scoring import league_points, STAT_COLUMNS

pd.set_option("display.width", 160)

## Building the three inputs `score_signal` needs, and a bug found along the way

`score_signal` wants a `signal` (one row per player-week with a `signal_value`), `actuals` (one row
per player-week with `actual_points`), and `projection` (the week's own Sleeper number). All three
are league-specific here: `defense_vs_position` and true fantasy points both depend on a league's
own scoring rules, and the two leagues this warehouse tracks score receptions differently (0.5 vs
1.0 PPR).

**`actuals`** is `weekly_stats`' raw counting stats scored through `league_points` — the same
stat-to-points arithmetic `defense_vs_position` itself is built from, so a player's actual points
here and the points his opponent allowed there are computed identically. `weekly_stats` currently
holds 2015-2025 only: 2026's feeds are switched off (#117), a gap the roadmap already flags and this
notebook is about to run straight into.

**`signal`** joins `defense_vs_position`'s recency column onto the player's own week through
`weekly_stats.opponent_team` — the same column `defense_vs_position` itself grouped on to become
`defense_team`, so this is the same identity, not a second lookup.

**`projection`** is `weekly_projections.sleeper_points`, restricted to the scoring basis that
matches each league (half-PPR for the Sleeper league, PPR for ESPN). It's built as a *left* join
onto the full player-week population rather than the bare `weekly_projections` rows: `score_signal`
inner-joins the projection frame before it ever reaches its per-baseline `dropna`, so if projection
has zero overlap with the population being tested — which, as the next section shows, it currently
does — a bare inner join would silently wipe out *every* baseline's result, not just the projection
baseline's own. A left join keeps the row with a null `sleeper_points`, which lets the two
walk-forward baselines (a player's own season-to-date and last-3 PPG) survive on their own
population exactly as `score_signal`'s own docstring says a stricter baseline's gaps should be
handled.

In [2]:
LEAGUES = q("SELECT * FROM league_settings")

STATS = q(f"""
    SELECT player_id, season, week, position, {", ".join(STAT_COLUMNS)}
    FROM weekly_stats
    WHERE season_type = 'REG' AND position IN ('QB', 'RB', 'WR', 'TE')
""")

RECENCY_COLUMNS = {
    "season_to_date": "points_allowed_per_game_season_to_date",
    "last3": "points_allowed_per_game_last3",
    "last5": "points_allowed_per_game_last5",
}
LEAGUE_SCORING = {"sleeper": "half_ppr", "espn": "ppr"}


def build_actuals(league_row):
    out = STATS[["player_id", "season", "week", "position"]].copy()
    out["actual_points"] = league_points(STATS, league_row)
    return out


def build_signal(league_key, column):
    return q(f"""
        SELECT ws.player_id, ws.season, ws.week,
               dvp.{column} AS signal_value, dvp.games_observed
        FROM weekly_stats ws
        JOIN defense_vs_position dvp
          ON dvp.league_key = ?
         AND dvp.season = ws.season AND dvp.week = ws.week
         AND dvp.defense_team = ws.opponent_team AND dvp.position = ws.position
        WHERE ws.season_type = 'REG' AND ws.position IN ('QB', 'RB', 'WR', 'TE')
    """, [league_key])


def build_projection(league_key, population):
    real = q(
        "SELECT player_id, season, week, sleeper_points FROM weekly_projections WHERE scoring = ?",
        [LEAGUE_SCORING[league_key]],
    )
    return population[["player_id", "season", "week"]].merge(
        real, on=["player_id", "season", "week"], how="left"
    )

### An actual bug, found by running a real multi-season sample through the harness

`weekly_backtest._score_group` clustered its significance test by `group.groupby("week")` — week
*number* alone, not `(season, week)`. Every test fixture in `test_weekly_backtest.py` used a single
season, so week numbers were already unique and the bug never showed: week 5 of 2015 and week 5 of
2024 were silently pooled into one "cluster," understating how many independent weeks actually
back a result by roughly 11x on this sample. Fixed in `weekly_backtest.py` (`_score_group` now
groups on `["season", "week"]`), with a regression test (`test_clustering_is_by_season_and_week_
not_week_number_alone`) added to `test_weekly_backtest.py` before the fix, per this repo's testing
rule. Every number below is post-fix.

In [3]:
results = []
for _, league in LEAGUES.iterrows():
    league_key = league["league_key"]
    actuals = build_actuals(league)
    projection = build_projection(league_key, actuals)
    for recency_name, column in RECENCY_COLUMNS.items():
        signal = build_signal(league_key, column)[["player_id", "season", "week", "signal_value"]]
        scored = score_signal(signal, actuals, projection)
        scored["league_key"] = league_key
        scored["recency"] = recency_name
        results.append(scored)

results = pd.concat(results, ignore_index=True)
results.shape

(66, 13)

<a id="q1"></a>
## 1. Does the as-of-week figure predict a player's points at all, per position?

Yes, for every position it can be measured on — QB, RB, TE and WR (no DST row exists at all; see
`defense_vs_position.py`'s own reasoning for why). The season-to-date column's raw correlation with
a player's actual points (`signal_rho`, no baseline held fixed) is small but consistently positive
and grows the same way through the position order the later questions use as their headline:

In [4]:
q1 = results[
    (results["recency"] == "season_to_date")
    & (results["baseline"] == "season_to_date_ppg")
    & (results["league_key"] == "sleeper")
][["position", "n", "n_weeks", "signal_rho"]].sort_values("position").reset_index(drop=True)
q1

,position,n,n_weeks,signal_rho
0,ALL,55525,181,0.008634
1,QB,6035,181,0.051268
2,RB,14710,181,0.036573
3,TE,11497,181,0.025350
4,WR,23283,181,0.014841


<a id="q2"></a>
## 2. Does it predict beyond the projection baseline?

No result exists to report: **n = 0** for every league, every recency window, every position. The
`sleeper_points` baseline needs a row where a completed game's actual points and a Sleeper weekly
projection both exist, and this warehouse currently has none — `weekly_stats` runs 2015-2025,
`weekly_projections`' `sleeper_points` only 2026. This is the sharpest confirmation yet of the
roadmap's #117 finding ("it is week 2 of 2026 and the warehouse holds zero 2026 rows"): it isn't
just that in-season feeds are switched off, it's that *no measurement holding the vendor projection
fixed can run at all* until they're switched back on. Re-run this section once #117 lands and a few
weeks of 2026 accumulate.

In [5]:
q2 = results[results["baseline"] == "sleeper_points"][
    ["league_key", "recency", "position", "n", "n_weeks"]
]
q2["n"].eq(0).all(), q2["n_weeks"].eq(0).all()

(True, True)

<a id="q3"></a>
## 3. Does a narrower recency window (last-3, last-5) beat the season-to-date figure?

No. Holding a player's own season-to-date PPG fixed, the season-to-date matchup column matches or
leads either recency window at every position, and for TE the last-3 window doesn't even clear
significance (p = 0.12) where season-to-date and last-5 both do (p < 0.001). There is no evidence
here that a shorter, noisier window on the defense side beats the fuller one.

In [6]:
q3 = pd.concat([
    results[(results["baseline"] == "season_to_date_ppg") & (results["league_key"] == "sleeper")
             & (results["position"] != "ALL")]
])[["recency", "position", "n", "n_weeks", "incremental_rho", "p_value"]]
q3.sort_values(["position", "recency"]).reset_index(drop=True)

,recency,position,n,n_weeks,incremental_rho,p_value
0,last3,QB,5322,159,0.074715,1.374692e-06
1,last5,QB,4568,137,0.085223,2.136775e-08
2,season_to_date,QB,6035,181,0.087805,1.293010e-11
3,last3,RB,12942,159,0.057863,4.704356e-09
4,last5,RB,11092,137,0.064421,2.299391e-09
5,season_to_date,RB,14710,181,0.064245,7.808446e-11
6,last3,TE,10220,159,0.015360,1.234968e-01
7,last5,TE,8747,137,0.039133,4.712114e-04
8,season_to_date,TE,11497,181,0.040316,8.468877e-05
9,last3,WR,20468,159,0.016230,1.566820e-02


<a id="q4"></a>
## 4. What is the minimum `games_observed` before the figure carries any signal?

Splitting the sample into disjoint buckets (1-2, 3-5, 6-9, 10-16 games observed) looks noisy and
inconsistent rather than a clean ramp: WR reads as insignificant through the first two buckets
(1-5 games), and TE clears significance in only one bucket of four (6-9 games), flipping back to
insignificant at 10-16. That non-monotonic wobble is bucket noise, not a real effect that comes and
goes — each disjoint slice is a much smaller, noisier sample than the full population it is carved
from.

In [7]:
BUCKETS = [(1, 2), (3, 5), (6, 9), (10, 16)]
league_key = "sleeper"
league = LEAGUES[LEAGUES["league_key"] == league_key].iloc[0]
actuals = build_actuals(league)
projection = build_projection(league_key, actuals)
signal_full = build_signal(league_key, RECENCY_COLUMNS["season_to_date"])

rows = []
for lo, hi in BUCKETS:
    bucketed = signal_full[signal_full["games_observed"].between(lo, hi)][
        ["player_id", "season", "week", "signal_value"]
    ]
    scored = score_signal(bucketed, actuals, projection)
    scored = scored[(scored["baseline"] == "season_to_date_ppg") & (scored["position"] != "ALL")]
    scored.insert(0, "games_observed", f"{lo}-{hi}")
    rows.append(scored[["games_observed", "position", "n", "n_weeks", "incremental_rho", "p_value"]])

pd.concat(rows).reset_index(drop=True)

,games_observed,position,n,n_weeks,incremental_rho,p_value
0,1-2,QB,713,22,0.072325,4.005540e-02
1,1-2,RB,1768,23,-0.022899,4.563967e-01
2,1-2,TE,1277,22,0.017938,5.412257e-01
3,1-2,WR,2815,23,0.028800,1.966756e-01
4,3-5,QB,1134,42,0.109590,2.607389e-03
5,3-5,RB,2824,44,0.096372,6.602904e-04
6,3-5,TE,2215,45,0.032976,2.427997e-01
7,3-5,WR,4473,44,-0.013142,4.025968e-01
8,6-9,QB,1565,55,0.066000,5.000381e-03
9,6-9,RB,3879,55,0.049691,8.586404e-03


A manager mid-season doesn't have an isolated disjoint slice, though — by week N a defense has
played *all* of its prior games, which is a cumulative population, not a bucket. Re-running the same
question as `games_observed >= k` for a range of `k` shows the effect already at essentially full
strength from `k = 0` (the lowest a non-null row can be) and not meaningfully growing with more
history, for every position:

In [8]:
THRESHOLDS = [0, 1, 2, 3, 5, 8, 10]
rows = []
for k in THRESHOLDS:
    filtered = signal_full[signal_full["games_observed"] >= k][
        ["player_id", "season", "week", "signal_value"]
    ]
    scored = score_signal(filtered, actuals, projection)
    scored = scored[(scored["baseline"] == "season_to_date_ppg") & (scored["position"] != "ALL")]
    scored.insert(0, "games_observed_at_least", k)
    rows.append(scored[["games_observed_at_least", "position", "n", "n_weeks", "incremental_rho", "p_value"]])

pd.concat(rows).reset_index(drop=True)

,games_observed_at_least,position,n,n_weeks,incremental_rho,p_value
0,0,QB,6035,181,0.087805,1.293010e-11
1,0,RB,14710,181,0.064245,7.808446e-11
2,0,TE,11497,181,0.040316,8.468877e-05
3,0,WR,23283,181,0.029802,5.227631e-05
4,1,QB,6035,181,0.087805,1.293010e-11
5,1,RB,14710,181,0.064245,7.808446e-11
6,1,TE,11497,181,0.040316,8.468877e-05
7,1,WR,23283,181,0.029802,5.227631e-05
8,2,QB,5681,170,0.089112,2.871938e-11
9,2,RB,13844,170,0.072840,6.775357e-13


No suppression threshold beyond what the column already enforces structurally — null until a
defense has played its first game — is supported by this data. A concrete cutoff like "3 games" or
"6 games" would be exactly the kind of guess this epic's own culture (`punt_environment.py`,
`draft_strategy.py`) already exists to refuse: the disjoint-bucket table above is a cautionary
example of a threshold that looks real and isn't.

<a id="q5"></a>
## 5. Does predictive value differ by position — and does it match the common TE/DST claim?

It differs by position, and it runs against the common claim rather than confirming it. Sorted by
effect size, holding a player's own season-to-date PPG fixed: **QB (0.088) > RB (0.064) > TE (0.040)
> WR (0.030)**, every one significant at p < 0.0005 on a sample in the tens of thousands, clustered
by 181 distinct season-weeks. The pooled "ALL" row across all four positions reads as flat and
insignificant in both leagues (incremental rho ~0.00 to -0.01, p > 0.14 either way) — that's not a
competing finding, it's an artifact of pooling positions that score fantasy points on entirely
different scales, and it's why every question above reports positions separately rather than
trusting the pooled row. DST isn't a computable row here at all (`defense_vs_position` has no
player rows to aggregate for it), so the "largest for TE/DST" half of the common claim can only be
checked on TE, and TE lands third of four, not first. Both leagues this warehouse tracks agree on
this ordering to within 0.002 of Spearman rho on every position, despite scoring receptions at
0.5 PPR and 1.0 PPR respectively — the finding isn't an artifact of either league's own scoring
rules.

In [9]:
q5 = results[
    (results["recency"] == "season_to_date") & (results["baseline"] == "season_to_date_ppg")
][["league_key", "position", "n", "n_weeks", "incremental_rho", "p_value", "ci_low", "ci_high"]]
q5.sort_values(["position", "league_key"]).reset_index(drop=True)

,league_key,position,n,n_weeks,incremental_rho,p_value,ci_low,ci_high
0,espn,ALL,55525,181,-0.005513,1.472080e-01,-0.012987,0.001960
1,sleeper,ALL,55525,181,-0.001467,6.960668e-01,-0.008866,0.005932
2,espn,QB,6035,181,0.088873,8.250471e-12,0.064895,0.112852
3,sleeper,QB,6035,181,0.087805,1.293010e-11,0.063858,0.111752
4,espn,RB,14710,181,0.062631,9.791764e-11,0.044656,0.080605
5,sleeper,RB,14710,181,0.064245,7.808446e-11,0.045916,0.082574
6,espn,TE,11497,181,0.042117,3.839652e-05,0.022432,0.061802
7,sleeper,TE,11497,181,0.040316,8.468877e-05,0.020538,0.060093
8,espn,WR,23283,181,0.031049,2.173935e-05,0.017001,0.045098
9,sleeper,WR,23283,181,0.029802,5.227631e-05,0.015614,0.043989


## Verdict

**Display-only.** Recorded in `defense_vs_position.py`'s module docstring in the
`punt_environment.py` "diagnostic rather than predictive" style — except this one isn't purely
diagnostic: the effect survives holding a player's own recent scoring fixed, at every measurable
position, on eleven seasons of data, confirmed across both leagues, with no games-observed
suppression threshold needed. It stays *display* rather than *weighted* because the actual
promotion bar this epic sets — beating the vendor's own weekly projection — is unanswerable right
now (question 2), not because the raw signal is weak. Worth re-running once #117's archive gives
this a season with both actuals and a live projection to hold fixed.